In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "your API token"

print("Kaggle API token set")


Kaggle API token set


In [ ]:
!kaggle competitions list | head


ref                                                                                 deadline             category                reward  teamCount  userHasEntered  
----------------------------------------------------------------------------------  -------------------  ---------------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3       2026-04-15 23:59:00  Featured         2,207,152 Usd       2000           False  
https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection            2026-02-27 23:59:00  Research           200,000 Usd       1329           False  
https://www.kaggle.com/competitions/stanford-rna-3d-folding-2                       2026-03-25 23:59:00  Featured           100,000 Usd        852           False  
https://www.kaggle.com/competitions/med-gemma-impact-challenge                      2026-02-24 23:59:00  Featured           100,000 Usd        124           False  
https://ww

In [ ]:
import os, json

# KAGGLE DETAILS
KAGGLE_USERNAME = "your kaggle username"
KAGGLE_KEY = "your API TOKEN"

# Create directory
os.makedirs("/root/.config/kaggle", exist_ok=True)

# Create kaggle.json
with open("/root/.config/kaggle/kaggle.json", "w") as f:
    json.dump({
        "username": KAGGLE_USERNAME,
        "key": KAGGLE_KEY
    }, f)

# Set correct permissions
os.chmod("/root/.config/kaggle/kaggle.json", 600)

print("kaggle.json created successfully")


kaggle.json created successfully


In [ ]:
!kaggle datasets list | head


ref                                                             title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  ------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
saidaminsaidaxmadov/chocolate-sales                             Chocolate Sales                                       468320  2026-01-04 14:23:35.490000          17542        312  1.0              
aliiihussain/amazon-sales-dataset                               Amazon_Sales_Dataset                                 1297759  2026-02-01 11:37:12.353000           3718         70  1.0              
ayeshasiddiqa123/student-perfirmance                            Student Academic Performance Dataset.                  96178  2026-01-06 12:08:32.540000           5978        110  1.0              
sidramazam

In [ ]:
!kaggle datasets download -d emmarex/plantdisease


Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
 94% 619M/658M [00:03<00:00, 63.4MB/s]
100% 658M/658M [00:03<00:00, 187MB/s] 


In [ ]:
from zipfile import ZipFile

zip_path = "/content/plantdisease.zip"
extract_path = "/content/dataset"

with ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully")


Dataset extracted successfully


In [ ]:
import os


os.listdir("/content/dataset/PlantVillage")[:10]


['Pepper__bell___Bacterial_spot',
 'Tomato__Target_Spot',
 'Tomato_healthy',
 'Potato___healthy',
 'Tomato_Spider_mites_Two_spotted_spider_mite',
 'Potato___Early_blight',
 'Tomato_Early_blight',
 'Tomato_Bacterial_spot',
 'Tomato_Leaf_Mold',
 'Tomato__Tomato_mosaic_virus']

In [ ]:
# ======================================
# CELL 1: TRAIN AND SAVE MODEL( this is using pytorch)
# ======================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
torch.manual_seed(42)


# ---- FIX RANDOMNESS ----
torch.manual_seed(42)

# ---- DEVICE ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- SETTINGS ----
base_dir = "/content/dataset/PlantVillage"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 7

# ---- TRANSFORMS (ImageNet normalization) ----
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],   # ImageNet mean
        [0.229, 0.224, 0.225]    # ImageNet std
    )
])

# ---- LOAD DATA ----
full_dataset = datasets.ImageFolder(base_dir, transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = full_dataset.classes

# ---- LOAD PRETRAINED MODEL ----
model = models.mobilenet_v2(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

for param in model.features[-3:].parameters():
    param.requires_grad = True


model.classifier[1] = nn.Sequential(
    nn.Linear(model.last_channel, 128),
    nn.ReLU(),
    nn.Linear(128, len(class_names))
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# ---- TRAIN ----
train_acc_history = []
val_acc_history = []

for epoch in range(EPOCHS):
    model.train()
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_accuracy = correct / total
    train_acc_history.append(train_accuracy)

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    val_accuracy = correct / total
    val_acc_history.append(val_accuracy)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Acc: {train_accuracy:.4f} "
          f"Val Acc: {val_accuracy:.4f}")

# ---- SAVE MODEL ----
torch.save(model.state_dict(), "plant_model.pth")

print("\n✅ Model trained and saved successfully!")


Epoch [1/7] Train Acc: 0.8242 Val Acc: 0.9271
Epoch [2/7] Train Acc: 0.9048 Val Acc: 0.9302
Epoch [3/7] Train Acc: 0.9156 Val Acc: 0.9532
Epoch [4/7] Train Acc: 0.9293 Val Acc: 0.9479
Epoch [5/7] Train Acc: 0.9295 Val Acc: 0.9557
Epoch [6/7] Train Acc: 0.9371 Val Acc: 0.9610
Epoch [7/7] Train Acc: 0.9387 Val Acc: 0.9564

✅ Model trained and saved successfully!


In [ ]:
# ======================================
# CELL 2: LOAD TRAINED MODEL
# ======================================

import torch
from torchvision import models, transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Same transform as training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# Rebuild model architecture
from torchvision import datasets

base_dir = "/content/dataset/PlantVillage"
full_dataset = datasets.ImageFolder(base_dir)
class_names = full_dataset.classes

model = models.mobilenet_v2(pretrained=False)

model.classifier[1] = torch.nn.Sequential(
    torch.nn.Linear(model.last_channel, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, len(class_names))
)

model.load_state_dict(torch.load("plant_model.pth"))
model = model.to(device)
model.eval()

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [ ]:
# ======================================
# CELL 3: UPLOAD AND PREDICT
# ======================================

from google.colab import files
from PIL import Image
import torch

def predict_image(image_path):
    img = Image.open(image_path).convert("RGB")
    img = transform(img)
    img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        probabilities = torch.softmax(outputs, dim=1)
        top3_prob, top3_idx = torch.topk(probabilities, 3)

    print("===================================")
    print("Top 3 Predictions:")

    for i in range(3):
        print(f"{class_names[top3_idx[0][i]]} → {top3_prob[0][i]*100:.2f}%")

    print("===================================")

# ---- Upload Images ----
uploaded = files.upload()

for file_name in uploaded.keys():
    predict_image(file_name)



Saving Screenshot_11-2-2026_173653_www.bing.com.jpeg to Screenshot_11-2-2026_173653_www.bing.com.jpeg
Top 3 Predictions:
Tomato__Tomato_YellowLeaf__Curl_Virus → 91.43%
Tomato_Early_blight → 6.64%
Tomato_Late_blight → 1.64%
